<a href="https://colab.research.google.com/github/sofilukash/math_modeling/blob/main/lab1/task3_atm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторна робота № 1. Завдання 3
## Банкомат (скінченний автомат за схемою на Рис. 1)

Кожен блок схеми $a_0\dots a_{13}$ реалізований окремим методом `state_aN` класу `ATM`.
Метод виконує дію блоку й **повертає ім'я наступного стану** — рівно так, як стрілки на схемі.

| Стан | Дія | Далі |
|---|---|---|
| a0 | запрошення до вводу картки | a1 |
| a1 | розпізнання картки | «+» → a2, «−» → a13 |
| a2 | запрошення до вводу коду | a3 |
| a3 | перевірка коду | «+» → a4, «−» → a11 |
| a4 | меню: стан рахунку / зняття / вихід | a5 / a6 / a10 |
| a5 | вивід стану рахунку | a4 |
| a6 | запит суми | a7 |
| a7 | перевірка наявності грошей | «+» → a8, «−» → a9 |
| a8 | видача готівки, зменшення рахунку | a4 |
| a9 | повідомлення «недостатньо коштів» | a4 |
| a10 | прощальні вітання | кінець |
| a11 | повторний ввід коду | «+» → a4, «−» → a12 |
| a12 | невірний код, блокування картки | кінець |
| a13 | тип картки не підтримується | кінець |

**Початкова сума рахунку** — випадкова: цілі гривні (0…20000) та цілі копійки (0…99).
**Картка** має цілі `тип` і `код`; банкомат обслуговує лише певні типи (`SUPPORTED_TYPES`) і коди (`ALLOWED_CODES`).

**Режими запуску:** `1` — демонстрація (готові сценарії), `2` — інтерактивна робота, `3` — обидва.

In [1]:
import random
from dataclasses import dataclass

# ---- Параметри банкомату (задаються попередньо) ----
SUPPORTED_TYPES = {1, 2, 3}                   # типи карток, які обслуговує банкомат
ALLOWED_CODES = {1234, 4321, 5555, 2026}      # коди, які банкомат визнає
BANKNOTES = [1000, 500, 200, 100, 50, 20, 10, 5, 2, 1]   # номінали, грн (кількість необмежена)


@dataclass
class Card:
    type: int            # тип картки (ціле)
    code: int            # код картки (ціле)
    blocked: bool = False


def fmt_money(kop):
    """Копійки (ціле) -> '123 грн 05 коп'."""
    return f"{kop // 100} грн {kop % 100:02d} коп"


def random_balance():
    """Випадкова початкова сума: цілі гривні та цілі копійки."""
    return random.randint(0, 20000) * 100 + random.randint(0, 99)


def parse_amount(text):
    """'150', '150.5', '150,50' -> копійки (int). Некоректне або ≤0 -> None."""
    try:
        s = str(text).strip().replace(",", ".")
        if s.count(".") > 1 or s == "":
            return None
        hrn, _, kop = s.partition(".")
        if (hrn and not hrn.isdigit()) or (kop and not kop.isdigit()) or len(kop) > 2:
            return None
        total = int(hrn or 0) * 100 + int((kop + "0")[:2] if kop else 0)
        return total if total > 0 else None
    except ValueError:
        return None


def parse_int(text):
    try:
        return int(str(text).strip())
    except ValueError:
        return None


def banknotes_for(kop):
    """Розклад суми на банкноти (жадібно); копійки — дрібними монетами."""
    hrn, rest = divmod(kop, 100)
    result = []
    for b in BANKNOTES:
        cnt, hrn = divmod(hrn, b)
        if cnt:
            result.append((b, cnt))
    return result, rest

## Клас `ATM` — автомат станів

In [2]:
class ATM:
    def __init__(self, balance_kop, card_provider, input_fn=input, print_fn=print):
        self.balance = balance_kop            # поточний стан рахунку, копійки
        self.card_provider = card_provider    # функція: повертає Card або None (вимкнути банкомат)
        self.inp = input_fn
        self.out = print_fn
        self.card = None
        self.entered_code = None
        self.reject_reason = ""
        self.state = "a0"
        self.trace = []
        self.powered_off = False

    # --- службове ---
    def say(self, text):
        self.out(f"  [{self.state}] {text}")

    def ask(self, prompt):
        return self.inp(f"  [{self.state}] {prompt}")

    def run_session(self):
        """Один сеанс: від a0 до кінцевого стану. Повертає послідовність пройдених станів."""
        self.trace = []
        self.state = "a0"
        while self.state is not None:
            self.trace.append(self.state)
            self.state = getattr(self, "state_" + self.state)()
            # (self.state оновлюється лише після виконання дії блоку)
        self.out("  Шлях за схемою: " + " → ".join(self.trace))
        return self.trace

    # --- блоки схеми ---
    def state_a0(self):
        self.say("Вставте картку.")
        self.card = self.card_provider()
        if self.card is None:                 # користувач вимкнув банкомат
            self.powered_off = True
            self.say("Банкомат вимкнено.")
            return None
        return "a1"

    def state_a1(self):
        # Розпізнання: тип має підтримуватись (+ розширення: заблокована картка не обслуговується)
        if self.card.blocked:
            self.reject_reason = "Картку заблоковано. Зверніться до банку."
            return "a13"
        if self.card.type not in SUPPORTED_TYPES:
            self.reject_reason = f"Тип картки {self.card.type} не підтримується цим банкоматом."
            return "a13"
        self.say(f"Картку розпізнано (тип {self.card.type}).")
        return "a2"

    def state_a2(self):
        self.entered_code = parse_int(self.ask("Введіть код картки: "))
        return "a3"

    def code_ok(self):
        return self.entered_code == self.card.code and self.card.code in ALLOWED_CODES

    def state_a3(self):
        return "a4" if self.code_ok() else "a11"

    def state_a4(self):
        self.say("МЕНЮ: 1 — стан рахунку, 2 — зняття готівки, 3 — вихід")
        choice = parse_int(self.ask("Ваш вибір: "))
        if choice == 1:
            return "a5"
        if choice == 2:
            return "a6"
        if choice == 3:
            return "a10"
        self.say("Невірний пункт меню, спробуйте ще раз.")
        return "a4"

    def state_a5(self):
        self.say(f"Стан рахунку: {fmt_money(self.balance)}")
        return "a4"

    def state_a6(self):
        self.amount = parse_amount(self.ask("Введіть суму для зняття (грн, напр. 150 або 150.50): "))
        if self.amount is None:
            self.say("Некоректна сума. Введіть додатне число (до двох знаків після коми).")
            return "a6"                        # повторний запит суми
        return "a7"

    def state_a7(self):
        return "a8" if self.amount <= self.balance else "a9"

    def state_a8(self):
        notes, coins = banknotes_for(self.amount)
        self.balance -= self.amount            # корегування рахунку
        text = ", ".join(f"{b} грн × {c}" for b, c in notes) or "—"
        self.say(f"Видано {fmt_money(self.amount)}. Банкноти: {text}" + (f"; дрібні: {coins} коп" if coins else ""))
        self.say(f"Залишок на рахунку: {fmt_money(self.balance)}")
        return "a4"

    def state_a9(self):
        self.say(f"Недостатньо коштів на рахунку (запит {fmt_money(self.amount)}, "
                 f"доступно {fmt_money(self.balance)}).")
        return "a4"

    def state_a10(self):
        self.say("Дякуємо, що скористались нашим банкоматом. До побачення!")
        return None

    def state_a11(self):
        self.say("Невірний код. Спробуйте ще раз (залишилась остання спроба).")
        self.entered_code = parse_int(self.ask("Повторно введіть код: "))
        return "a4" if self.code_ok() else "a12"

    def state_a12(self):
        self.card.blocked = True
        self.say("Невірний код. Картку заблоковано.")
        return None

    def state_a13(self):
        self.say(self.reject_reason)
        return None

## Демонстрація: готові сценарії з перевіркою шляху за схемою
Вводи користувача імітуються (виводяться в тому ж вигляді, ніби їх надрукували).

In [3]:
def scripted(inputs):
    """Імітація клавіатури: повертає наступний заготовлений ввід і друкує його."""
    it = iter(inputs)

    def fn(prompt=""):
        value = next(it)
        print(f"{prompt}{value}")
        return value
    return fn


def run_scenario(title, card, inputs, balance_kop, expected_path):
    print("\n" + "=" * 78)
    print(f"СЦЕНАРІЙ: {title}")
    print("=" * 78)
    cards = iter([card])
    atm = ATM(balance_kop, lambda: next(cards, None), input_fn=scripted(inputs))
    print(f"  Початковий стан рахунку: {fmt_money(balance_kop)}")
    trace = atm.run_session()
    assert trace == expected_path, f"Шлях не збігся зі схемою!\n  є:       {trace}\n  очікували: {expected_path}"
    print("  ✔ Шлях збігається зі схемою Рис. 1")
    return atm


def demo():
    # 1. Непідтримуваний тип картки: a0 → a1 → a13
    run_scenario("непідтримуваний тип картки", Card(type=9, code=1234), [], 100000,
                 ["a0", "a1", "a13"])

    # 2. Невірний код, потім вірний; стан рахунку, зняття, спроба зняти забагато, вихід
    atm = run_scenario("помилка коду з другої спроби, операції з рахунком", Card(type=1, code=1234),
                       ["1111",        # a2: невірний код  -> a3 -> a11
                        "1234",        # a11: вірний код   -> a4
                        "1",           # a4: стан рахунку  -> a5 -> a4
                        "2", "150.50", # a4: зняття; a6: сума 150.50 -> a7 -> a8 -> a4
                        "2", "5000",   # зняття 5000 > залишку -> a7 -> a9 -> a4
                        "3"],          # вихід -> a10
                       100000,         # на рахунку 1000.00 грн
                       ["a0", "a1", "a2", "a3", "a11", "a4", "a5", "a4", "a6", "a7", "a8", "a4",
                        "a6", "a7", "a9", "a4", "a10"])
    assert atm.balance == 100000 - 15050, "Залишок після зняття 150.50 має бути 849.50"
    print(f"  ✔ Залишок {fmt_money(atm.balance)} (1000.00 − 150.50 = 849.50)")

    # 3. Два невірні коди -> блокування; повторне вставлення тієї ж картки
    card = Card(type=2, code=4321)
    run_scenario("два невірні коди → блокування картки", card, ["0000", "1111"], 5000,
                 ["a0", "a1", "a2", "a3", "a11", "a12"])
    run_scenario("повторне вставлення заблокованої картки", card, [], 5000, ["a0", "a1", "a13"])

    # 4. Випадкова початкова сума (як вимагає завдання) + некоректний ввід суми й пункту меню
    bal = random_balance()
    enough = bal >= 100                       # знімаємо 1 грн: якщо рахунок < 1 грн, буде гілка a9
    path = ["a0", "a1", "a2", "a3", "a4", "a4", "a4", "a6", "a6", "a6", "a7",
            "a8" if enough else "a9", "a4", "a10"]
    atm = run_scenario("випадкова початкова сума, некоректні вводи (меню: 7, abc; сума: -5, x)",
                       Card(type=3, code=5555),
                       ["5555", "7", "abc", "2", "-5", "x", "1", "3"], bal, path)
    assert atm.balance == (bal - 100 if enough else bal)
    print(f"  ✔ Залишок {fmt_money(atm.balance)}")
    print("\nДемонстрацію завершено: усі сценарії відповідають схемі.")

## Інтерактивний режим
Вибір «картки» з гаманця (або власні тип та код). Початкова сума — випадкова.
Картка заблокована після двох невірних кодів — повторно вона вже не прийметься.

In [4]:
def interactive():
    wallet = [Card(1, 1234), Card(2, 4321), Card(7, 1111)]
    balance = random_balance()
    print("\nІНТЕРАКТИВНИЙ РЕЖИМ")
    print("(Початкова сума рахунку — випадкова; побачити її можна через меню банкомату.)")
    print("Підказка для тесту: картка 1 — код 1234, картка 2 — код 4321, картка 3 — тип 7 (не підтримується).")

    def provider():
        print("\n  Гаманець: " + "; ".join(f"{i + 1}) тип {c.type}{' [заблок.]' if c.blocked else ''}"
                                          for i, c in enumerate(wallet)) + "; 0) своя картка; q) вимкнути")
        s = input("  Оберіть картку: ").strip().lower()
        if s == "q":
            return None
        if s == "0":
            t, c = parse_int(input("  Тип картки: ")), parse_int(input("  Код картки: "))
            card = Card(t if t is not None else -1, c if c is not None else -1)
            wallet.append(card)
            return card
        idx = parse_int(s)
        if idx is not None and 1 <= idx <= len(wallet):
            return wallet[idx - 1]
        print("  Такої картки немає.")
        return provider()

    atm = ATM(balance, provider)
    while not atm.powered_off:
        atm.run_session()
        print()

In [5]:
def main():
    print("Режим: 1 — демонстрація, 2 — інтерактивний, 3 — обидва")
    mode = input("Ваш вибір [1]: ").strip() or "1"
    if mode in ("1", "3"):
        demo()
    if mode in ("2", "3"):
        interactive()


if __name__ == "__main__":
    main()

Режим: 1 — демонстрація, 2 — інтерактивний, 3 — обидва
Ваш вибір [1]: 1

СЦЕНАРІЙ: непідтримуваний тип картки
  Початковий стан рахунку: 1000 грн 00 коп
  [a0] Вставте картку.
  [a13] Тип картки 9 не підтримується цим банкоматом.
  Шлях за схемою: a0 → a1 → a13
  ✔ Шлях збігається зі схемою Рис. 1

СЦЕНАРІЙ: помилка коду з другої спроби, операції з рахунком
  Початковий стан рахунку: 1000 грн 00 коп
  [a0] Вставте картку.
  [a1] Картку розпізнано (тип 1).
  [a2] Введіть код картки: 1111
  [a11] Невірний код. Спробуйте ще раз (залишилась остання спроба).
  [a11] Повторно введіть код: 1234
  [a4] МЕНЮ: 1 — стан рахунку, 2 — зняття готівки, 3 — вихід
  [a4] Ваш вибір: 1
  [a5] Стан рахунку: 1000 грн 00 коп
  [a4] МЕНЮ: 1 — стан рахунку, 2 — зняття готівки, 3 — вихід
  [a4] Ваш вибір: 2
  [a6] Введіть суму для зняття (грн, напр. 150 або 150.50): 150.50
  [a8] Видано 150 грн 50 коп. Банкноти: 100 грн × 1, 50 грн × 1; дрібні: 50 коп
  [a8] Залишок на рахунку: 849 грн 50 коп
  [a4] МЕНЮ: 1 — 

## Відповідь на контрольне питання 1.5.3
**Так.** Мова Python дозволяє організувати повторний ввід коду без окремого стану $a_{11}$ — за допомогою циклу:

```python
attempts = 0
while attempts < 2:                  # дві спроби
    code = int(input("Код: "))
    if code == card.code:
        break                        # → меню a4
    attempts += 1
else:
    card.blocked = True              # → a12 (блокування)
```

Тут роль $a_{11}$ виконує тіло циклу `while` (лічильник `attempts`), а умова виходу з циклу — перехід до $a_4$ або $a_{12}$.
Також можна використати рекурсію або генератор, але цикл — найпростіший спосіб.

## Висновки (завдання 3)
* Алгоритм банкомату реалізовано як скінченний автомат: кожен блок $a_0\dots a_{13}$ — окремий метод,
  що повертає наступний стан, тому код безпосередньо відповідає схемі Рис. 1.
* Демонстрація перевіряє (через `assert`) послідовність станів для всіх гілок схеми:
  непідтримувана картка ($a_{13}$), помилка коду з відновленням ($a_{11}\to a_4$), блокування ($a_{12}$),
  стан рахунку ($a_5$), зняття ($a_8$), нестача коштів ($a_9$), вихід ($a_{10}$).
* Початкова сума рахунку — випадкова, з цілими гривнями та копійками; усі розрахунки ведуться в цілих копійках,
  тому помилок округлення немає.